# Convert `sensix-zo/nllb-paite-600m-v15` → CTranslate2 (int8)

Run on **Google Colab** with GPU runtime (faster download/conversion).

1. Set your HF token in the next cell (or use Colab Secrets)
2. Run all cells
3. Creates repo `sensix-zo/nllb-paite-600m-v15-ct2` and uploads the converted model
4. Then on Coolify set:
   - `INFERENCE_BACKEND=ctranslate2`
   - `CT2_MODEL_REPO=sensix-zo/nllb-paite-600m-v15-ct2`
   - `CT2_COMPUTE_TYPE=int8`

In [ ]:
# Install once
!pip install -q ctranslate2 transformers sentencepiece huggingface_hub accelerate

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

from huggingface_hub import HfApi, hf_hub_download, snapshot_download, login

# Paste token here OR: from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')
HF_TOKEN = "PASTE_YOUR_HF_TOKEN_HERE"

MODEL_REPO = "sensix-zo/nllb-paite-600m-v15"
BASE_NLLB_REPO = "facebook/nllb-200-distilled-600M"
CT2_REPO = "sensix-zo/nllb-paite-600m-v15-ct2"
QUANTIZATION = "int8"  # best for CPU VPS

WORK = Path("/content/paite-ct2")
HF_STAGING = WORK / "hf-model"
CT2_OUT = WORK / "ct2-model"

if HF_TOKEN == "PASTE_YOUR_HF_TOKEN_HERE":
    raise ValueError("Set HF_TOKEN first")

login(token=HF_TOKEN)
print("Logged in to Hugging Face")

In [ ]:
# Download finetuned PyTorch model
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

print(f"Downloading {MODEL_REPO}...")
local_repo = snapshot_download(
    repo_id=MODEL_REPO,
    token=HF_TOKEN,
    local_dir=str(HF_STAGING),
    local_dir_use_symlinks=False,
)

# NLLB needs sentencepiece from base model
spm_path = hf_hub_download(
    repo_id=BASE_NLLB_REPO,
    filename="sentencepiece.bpe.model",
    token=HF_TOKEN,
)
shutil.copy(spm_path, Path(local_repo) / "sentencepiece.bpe.model")
print("Model downloaded:", local_repo)

In [ ]:
# Convert to CTranslate2
if CT2_OUT.exists():
    shutil.rmtree(CT2_OUT)

cmd = [
    "ct2-transformers-converter",
    "--model", str(HF_STAGING),
    "--output_dir", str(CT2_OUT),
    "--quantization", QUANTIZATION,
    "--force",
    "--copy_files", "tokenizer_config.json",
    "--copy_files", "sentencepiece.bpe.model",
    "--copy_files", "special_tokens_map.json",
]
if (HF_STAGING / "tokenizer.json").exists():
    cmd.extend(["--copy_files", "tokenizer.json"])

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("\nConverted files:")
!ls -lh {CT2_OUT}

In [ ]:
# Quick sanity check
import ctranslate2
from transformers import NllbTokenizer

translator = ctranslate2.Translator(str(CT2_OUT), device="cpu", compute_type=QUANTIZATION)
tokenizer = NllbTokenizer.from_pretrained(str(CT2_OUT))

src_lang, tgt_lang = "eng_Latn", "pai_Latn"
text = "Hello, how are you today?"
tokenizer.src_lang = src_lang
source = tokenizer.convert_ids_to_tokens(tokenizer.encode(text))
result = translator.translate_batch([source], target_prefix=[[tgt_lang]])
out_tokens = result[0].hypotheses[0][1:]
print("EN:", text)
print("PAITE:", tokenizer.decode(tokenizer.convert_tokens_to_ids(out_tokens)))

In [ ]:
# Create HF repo and upload (one-time)
api = HfApi(token=HF_TOKEN)

try:
    api.create_repo(CT2_REPO, repo_type="model", exist_ok=True)
    print(f"Repo ready: https://huggingface.co/{CT2_REPO}")
except Exception as e:
    print("Repo note:", e)

api.upload_folder(
    folder_path=str(CT2_OUT),
    repo_id=CT2_REPO,
    repo_type="model",
    commit_message="Add CTranslate2 int8 conversion for faster CPU inference",
)

print(f"\nDone! Model uploaded: https://huggingface.co/{CT2_REPO}")
print("\nCoolify env:")
print("  INFERENCE_BACKEND=ctranslate2")
print(f"  CT2_MODEL_REPO={CT2_REPO}")
print("  CT2_COMPUTE_TYPE=int8")